# Crawl Dữ Liệu Việc Làm CNTT từ TopCV (Trang 1 - 46)

> **Lưu ý quan trọng về Cloudflare:**
> TopCV áp dụng hệ thống bảo vệ Cloudflare (mã lỗi 403 khi dùng `requests` thông thường).
> Code sử dụng thư viện **`curl_cffi`** với tính năng giả lập TLS Fingerprint của trình duyệt Chrome (`impersonate="chrome124"`) để vượt qua Cloudflare một cách trơn tru và an toàn.

In [5]:
# Cài đặt thư viện nếu chưa có:
# !pip install curl_cffi beautifulsoup4 pandas tqdm

import os
import time
import random
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests
from tqdm import tqdm

In [6]:
def crawl_topcv_jobs(start_page: int = 1, end_page: int = 46, delay_range: tuple = (1.5, 3.0)) -> pd.DataFrame:
    """
    Crawl danh sách việc làm CNTT từ TopCV.
    
    Thông tin trích xuất cho mỗi job:
    - source: 'topcv'
    - source_job_id: ID định danh của job trên TopCV
    - job_url: Link chi tiết công việc
    - company_logo_url: Link ảnh logo công ty
    - tile_raw: Tiêu đề công việc gốc
    - company_name_raw: Tên công ty tuyển dụng
    - location_raw: Địa điểm làm việc
    - salary_raw: Mức lương hiển thị
    - experience_raw: Yêu cầu kinh nghiệm
    - posted_at_raw: Ngày đăng / cập nhật
    - crawled_at: Thời điểm crawl dữ liệu (ISO format)
    """
    base_url = "https://www.topcv.vn/tim-viec-lam-cong-nghe-thong-tin-cr257?type_keyword=1&disable_auto_detect_type_keyword=1&page={page}&category_family=r257&saturday_status=0&sba=1"
    
    # Sử dụng Session với TLS Impersonate để vượt Cloudflare
    session = requests.Session(impersonate="chrome124")
    
    all_jobs = []
    
    print(f"[*] Bắt đầu crawl từ page {start_page} đến page {end_page}...")
    
    for page in tqdm(range(start_page, end_page + 1), desc="Đang cào dữ liệu TopCV"):
        url = base_url.format(page=page)
        max_retries = 3
        page_jobs = []
        
        for attempt in range(max_retries):
            try:
                resp = session.get(url, timeout=20)
                if resp.status_code == 200:
                    soup = BeautifulSoup(resp.text, "html.parser")
                    cards = soup.select(".job-item-search-result")
                    
                    for card in cards:
                        # 1. Job ID
                        job_id = card.get("data-job-id", "").strip()
                        
                        # 2. Tiêu đề & URL
                        title_el = card.select_one(".title a, h3 a, .job-title a")
                        tile_raw = title_el.get_text(strip=True) if title_el else ""
                        raw_url = title_el.get("href", "") if title_el else ""
                        job_url = raw_url.split("?")[0] if raw_url else ""  # Chuẩn hoá URL, bỏ query string
                        
                        # 3. Logo công ty / Link ảnh (TopCV dùng lazy load với data-src)
                        img_el = card.select_one(".avatar img, .company-logo img, img")
                        company_logo_url = ""
                        if img_el:
                            company_logo_url = img_el.get("data-src") or img_el.get("src") or ""
                        
                        # 4. Tên công ty
                        company_el = card.select_one(".company-name, a.company, .company")
                        company_name_raw = company_el.get_text(strip=True) if company_el else ""
                        
                        # 5. Mức lương
                        salary_el = card.select_one(".salary")
                        salary_raw = salary_el.get_text(strip=True) if salary_el else ""
                        
                        # 6. Kinh nghiệm
                        exp_el = card.select_one(".exp")
                        experience_raw = exp_el.get_text(strip=True) if exp_el else ""
                        
                        # 7. Địa điểm
                        loc_el = card.select_one(".address.truncate, .city-text, .address")
                        location_raw = loc_el.get_text(strip=True) if loc_el else ""
                        
                        # 8. Thời gian đăng
                        date_el = card.select_one(".label-update, .time")
                        posted_at_raw = date_el.get_text(strip=True) if date_el else ""
                        
                        page_jobs.append({
                            "source": "topcv",
                            "source_job_id": job_id,
                            "job_url": job_url,
                            "company_logo_url": company_logo_url,
                            "tile_raw": tile_raw,
                            "company_name_raw": company_name_raw,
                            "location_raw": location_raw,
                            "salary_raw": salary_raw,
                            "experience_raw": experience_raw,
                            "posted_at_raw": posted_at_raw,
                            "crawled_at": datetime.now().isoformat()
                        })
                    break
                else:
                    print(f"[!] Page {page} trả về mã lỗi HTTP {resp.status_code}. Thử lại {attempt + 1}/{max_retries}...")
                    time.sleep(2 * (attempt + 1))
            except Exception as e:
                print(f"[!] Lỗi khi cào page {page} (Lần thử {attempt + 1}): {e}")
                time.sleep(2 * (attempt + 1))
        
        all_jobs.extend(page_jobs)
        
        # Nghỉ ngẫu nhiên giữa các request để bảo vệ IP và tránh bị chặn
        time.sleep(random.uniform(*delay_range))
        
    df = pd.DataFrame(all_jobs)
    print(f"[+] Hoàn thành! Đã thu thập được tổng cộng {len(df)} việc làm.")
    return df

In [7]:
# Chạy crawl (bạn có thể đổi end_page thành 2 hoặc 5 để test trước khi chạy hết 46 trang)
df_jobs = crawl_topcv_jobs(start_page=1, end_page=46)

# Kiểm tra dữ liệu thu thập được
print("Shape:", df_jobs.shape)
df_jobs.head()

[*] Bắt đầu crawl từ page 1 đến page 46...


Đang cào dữ liệu TopCV:  54%|█████▍    | 25/46 [02:10<01:50,  5.25s/it]

[!] Page 26 trả về mã lỗi HTTP 429. Thử lại 1/3...
[!] Page 26 trả về mã lỗi HTTP 429. Thử lại 2/3...


Đang cào dữ liệu TopCV:  57%|█████▋    | 26/46 [02:22<02:30,  7.51s/it]

[!] Page 27 trả về mã lỗi HTTP 429. Thử lại 1/3...
[!] Page 27 trả về mã lỗi HTTP 429. Thử lại 2/3...


...

Đang cào dữ liệu TopCV:  98%|█████████▊| 45/46 [05:32<00:09,  9.80s/it]

[!] Page 46 trả về mã lỗi HTTP 429. Thử lại 1/3...
[!] Page 46 trả về mã lỗi HTTP 429. Thử lại 2/3...
[!] Page 46 trả về mã lỗi HTTP 429. Thử lại 3/3...


Đang cào dữ liệu TopCV: 100%|██████████| 46/46 [05:47<00:00,  7.55s/it]

[+] Hoàn thành! Đã thu thập được tổng cộng 1900 việc làm.
Shape: (1900, 11)


,source,source_job_id,job_url,company_logo_url,tile_raw,company_name_raw,location_raw,salary_raw,experience_raw,posted_at_raw,crawled_at
0,topcv,2301984,https://www.topcv.vn/viec-lam/ky-su-lap-trinh-...,https://cdn-new.topcv.vn/unsafe/150x/https://s...,Kỹ Sư Lập Trình CNC (Thu Nhập UpTo 25Tr/ Tháng),Công ty Cổ phần Apillis Việt Nam,Hồ Chí Minh (mới),18 - 25 triệu,2 năm,Đăng1 tuần trước,2026-10-07T22:21:36.318729
1,topcv,2319343,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,https://cdn-new.topcv.vn/unsafe/150x/https://s...,Nhân Viên Kinh Doanh - Web/App - Lương Cứng Kh...,CÔNG TY TNHH GIẢI PHÁP CÔNG NGHỆ SỐ VN,Hồ Chí Minh (mới),10 - 30 triệu,Không yêu cầu,Đăng5 ngày trước,2026-10-07T22:21:36.319728
2,topcv,2316286,https://www.topcv.vn/brand/misa/tuyen-dung/nha...,https://cdn-new.topcv.vn/unsafe/150x/https://s...,Nhân Viên Kinh Doanh/ Sale Giải Pháp Phần Mềm ...,ProCông ty Cổ phần MISA,Hồ Chí Minh (mới),Tới 20 triệu,Không yêu cầu,Đăng1 tuần trước,2026-10-07T22:21:36.320728
3,topcv,2313041,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,https://cdn-new.topcv.vn/unsafe/150x/https://s...,Nhân Viên Kinh Doanh/ Sales Online Phần Mềm Kế...,CÔNG TY CỔ PHẦN VACOM,Hà Nội,10 - 50 triệu,Không yêu cầu,Đăng1 tuần trước,2026-10-07T22:21:36.321727
4,topcv,2316742,https://www.topcv.vn/viec-lam/senior-developer...,https://cdn-new.topcv.vn/unsafe/150x/https://s...,"Senior Developer (NodeJS, ReactJS, AI)",CÔNG TY TNHH DEKON VIỆT NAM,Hồ Chí Minh (mới),Thoả thuận,4 năm,Đăng5 ngày trước,2026-10-07T22:21:36.322731


In [8]:
# Lưu dữ liệu vào thư mục data/raw
output_dir = os.path.join("..", "data", "raw")
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "topcv_it_jobs_raw.csv")
json_path = os.path.join(output_dir, "topcv_it_jobs_raw.json")

df_jobs.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_jobs.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"[+] Đã lưu file CSV: {csv_path}")
print(f"[+] Đã lưu file JSON: {json_path}")

[+] Đã lưu file CSV: ..\data\raw\topcv_it_jobs_raw.csv
[+] Đã lưu file JSON: ..\data\raw\topcv_it_jobs_raw.json


# Phần 2: Crawl Chi Tiết Từng Job (Job Details)

Crawl vào từng `job_url` đã thu thập từ danh sách để lấy thông tin chi tiết đầy đủ của từng vị trí tuyển dụng.

### Cấu trúc schema dữ liệu `job_details`:
```text
job_details
├── detail_id (VD: topcv_2301984)
├── source (topcv)
├── source_job_id
├── job_url
│
├── title_raw
│
├── company_name_raw
├── company_url
├── company_size_raw
├── company_location_raw
├── industry_raw
│
├── salary_raw
├── experience_raw
├── education_raw
├── location_raw
├── work_mode_raw
├── employment_type_raw
├── working_time_raw
│
├── overview_raw
├── responsibilities_raw
├── requirements_raw
├── benefits_raw
│
├── required_skills_raw
├── preferred_skills_raw
│
├── published_at_raw
├── updated_at_raw
├── deadline_raw
│
└── crawled_at
```

> **Tính năng an toàn khi crawl số lượng lớn (1900 jobs):**
> - **Checkpoint / Resume**: Tự động lưu tiến trình định kỳ sau mỗi N jobs; nếu tạm dừng giữa chừng khi chạy lại sẽ tự động tiếp tục từ job chưa cào.
> - **Chống 429 (Too Many Requests)**: Tự động giãn cách nghỉ ngẫu nhiên (1.5s - 3.0s) và retry backoff nếu gặp giới hạn tần suất.

In [12]:
import re
import json
def clean_val(val: str) -> str:
    """Xóa các ký tự thừa ở đầu/cuối như dấu hai chấm, dấu gạch nối."""
    if not val:
        return ""
    val = val.strip()
    val = re.sub(r"^[:\-\s]+", "", val)
    return val.strip()

def parse_job_detail(job_item: dict, html: str) -> dict:
    """
    Phân tích mã HTML của trang chi tiết job và trích xuất toàn bộ các trường schema.
    Tương thích với cả giao diện việc làm thông thường và giao diện thương hiệu (/brand/).
    """
    soup = BeautifulSoup(html, "html.parser")
    
    source_job_id = str(job_item.get("source_job_id", "")).strip()
    job_url = job_item.get("job_url", "")
    detail_id = f"topcv_{source_job_id}" if source_job_id else ""

    # 1. Title
    title = ""
    if "/brand/" in job_url:
        brand_title = soup.select_one(".premium-job-basic-information h2, .job-header h2, .premium-job-title")
        if brand_title:
            title = brand_title.get_text(separator=" ", strip=True)
    
    if not title:
        title_el = soup.select_one(
            ".job-detail__info--title, h1.job-detail-title, .box-header-job h1, "
            ".job-title, h1.title"
        )
        if title_el:
            title = title_el.get_text(separator=" ", strip=True)
            
    if not title or len(title) < 3:
        title = job_item.get("tile_raw", "")

    # 2. Company Name & URL
    comp_name = ""
    comp_url = ""
    comp_el = soup.select_one(
        ".company-name-label a, .box-company-info-detail__header a, "
        ".box-company-info__detail a.name, a.company-name, "
        ".premium-company-name a, .company-name a"
    )
    if comp_el:
        comp_name = comp_el.get_text(strip=True)
        comp_url = comp_el.get("href", "")
    
    if not comp_url and "/brand/" in job_url:
        brand_match = re.search(r"(https://www.topcv.vn/brand/[^/]+)", job_url)
        if brand_match:
            comp_url = brand_match.group(1)
            
    if not comp_name:
        comp_name = job_item.get("company_name_raw", "")
    if comp_name.startswith("ProCông ty"):
        comp_name = comp_name.replace("ProCông ty", "Công ty")

    # 3. Company attributes (quy mô, địa chỉ, ngành nghề)
    comp_size = ""
    comp_loc = ""
    industry = ""

    for it in soup.select(".box-company-info-detail__list--item, .box-general-group-item, .company-info-item"):
        txt = it.get_text(separator=": ", strip=True)
        if "Quy mô" in txt:
            comp_size = clean_val(txt.split(":", 1)[-1])
        elif "Địa điểm" in txt:
            comp_loc = clean_val(txt.split(":", 1)[-1])
        elif "Ngành nghề" in txt:
            industry = clean_val(txt.split(":", 1)[-1])

    if not industry:
        ind_el = soup.select_one(".box-company-info-detail__industries--text, .company-industry")
        if ind_el:
            industry = clean_val(ind_el.get_text(strip=True).replace("Ngành nghề:", ""))

    # 4. Job metadata: salary, experience, education, work_mode, employment_type, location
    salary = job_item.get("salary_raw", "")
    exp = job_item.get("experience_raw", "")
    edu = ""
    work_mode = ""
    emp_type = ""
    deadline = ""
    location = job_item.get("location_raw", "")

    header_box = soup.select_one(".box-header-job, .premium-job-basic-information")
    if header_box:
        header_text = header_box.get_text(separator=" | ", strip=True)
        sal_match = re.search(r"(?:Mức lương|Thu nhập)\s*\|\s*([^|]+)", header_text)
        if sal_match:
            salary = sal_match.group(1).strip()
        exp_match = re.search(r"Kinh nghiệm\s*\|\s*([^|]+)", header_text)
        if exp_match:
            exp = exp_match.group(1).strip()

    # Trích xuất thông tin chung (Học vấn, hình thức, loại hình làm việc)
    for div in soup.select(".box-job-information-general-info, .premium-job-general-information, .box-general"):
        txt = div.get_text(separator=" | ", strip=True)
        if "Học vấn" in txt and not edu:
            m = re.search(r"Học vấn\s*\|\s*([^|]+)", txt)
            if m: edu = m.group(1).strip()
        if "Hình thức làm việc" in txt and not work_mode:
            m = re.search(r"Hình thức làm việc\s*\|\s*([^|]+)", txt)
            if m: work_mode = m.group(1).strip()
        if "Loại hình làm việc" in txt and not emp_type:
            m = re.search(r"Loại hình làm việc\s*\|\s*([^|]+)", txt)
            if m: emp_type = m.group(1).strip()

    # Fallback tìm kiếm trong body
    if not (edu and work_mode and emp_type):
        body_text = soup.get_text(separator=" | ", strip=True)
        if not edu:
            m = re.search(r"Học vấn\s*\|\s*([^|]+)", body_text)
            if m: edu = m.group(1).strip()
        if not work_mode:
            m = re.search(r"Hình thức làm việc\s*\|\s*([^|]+)", body_text)
            if m: work_mode = m.group(1).strip()
        if not emp_type:
            m = re.search(r"Loại hình làm việc\s*\|\s*([^|]+)", body_text)
            if m: emp_type = m.group(1).strip()

    # Deadline & Dates
    dl_match = re.search(r"Hạn\s*(?:ứng tuyển|nộp hồ sơ)[:\s|]*([0-9]{1,2}/[0-9]{1,2}/[0-9]{4})", soup.get_text())
    if dl_match:
        deadline = dl_match.group(1)

    published_at = job_item.get("posted_at_raw", "")
    updated_at = ""
    pub_match = re.search(r"(?:Ngày đăng|Đăng ngày)[:\s|]*([0-9]{1,2}/[0-9]{1,2}/[0-9]{4})", soup.get_text())
    if pub_match:
        published_at = pub_match.group(1)
    upd_match = re.search(r"(?:Cập nhật|Ngày làm mới)[:\s|]*([0-9]{1,2}/[0-9]{1,2}/[0-9]{4})", soup.get_text())
    if upd_match:
        updated_at = upd_match.group(1)

    # 5. Working time
    working_time = ""
    for item in soup.select(".box-job-information-address-and-time-list__item, .premium-job-description__box"):
        t_el = item.find(["h2", "h3", "h4", "div", "span"])
        if t_el and "thời gian làm việc" in t_el.get_text().lower():
            content_el = item.select_one(".box-job-information-address-and-time-list__item--content")
            if content_el:
                working_time = content_el.get_text(separator=" ", strip=True)
            else:
                working_time = item.get_text(separator=" ", strip=True).replace(t_el.get_text(), "").strip()
            break

    # 6. Detailed content sections (overview, responsibilities, requirements, benefits)
    overview_raw = ""
    responsibilities_raw = ""
    requirements_raw = ""
    benefits_raw = ""

    section_boxes = soup.select(
        ".box-job-information-detail-item, .premium-job-description__box, "
        ".job-description__item, .job-detail-section"
    )
    for box in section_boxes:
        h = box.find(["h2", "h3", "h4", "div"], class_=re.compile(r"title|header|name"))
        if not h:
            continue
        h_text = h.get_text(strip=True).lower()

        content = box.select_one(
            ".box-job-information-detail-item__content, "
            ".box-job-information-detail-item__text, .content"
        )
        raw_c = content.get_text(separator="\n", strip=True) if content else box.get_text(separator="\n", strip=True)

        lines = [line.strip() for line in raw_c.split("\n") if line.strip() and line.strip().lower() != h_text]
        clean_section = "\n".join(lines)

        if "mô tả công việc" in h_text and not responsibilities_raw:
            responsibilities_raw = clean_section
        elif any(k in h_text for k in ["yêu cầu ứng viên", "yêu cầu công việc", "yêu cầu"]) and not requirements_raw:
            requirements_raw = clean_section
        elif any(k in h_text for k in ["quyền lợi", "phúc lợi"]) and not benefits_raw:
            benefits_raw = clean_section
        elif any(k in h_text for k in ["tổng quan", "mô tả chung", "về công việc"]) and not overview_raw:
            overview_raw = clean_section

    # 7. Skills (required & preferred)
    skills = []
    for s in soup.select(".job-tags .search-from-tag, .job-tags__group-list-tag a, .box-category-tag"):
        t = s.get_text(strip=True)
        if t and t not in skills:
            skills.append(t)
    required_skills_raw = ", ".join(skills)
    
    preferred_skills_raw = ""
    if requirements_raw:
        pref_match = re.search(r"(?:Ưu tiên|Điểm cộng|Yêu cầu ưu tiên)[:\s]*([^\n]+(?:\n[^\n]+)*)", requirements_raw, re.IGNORECASE)
        if pref_match:
            preferred_skills_raw = pref_match.group(1).strip()

    return {
        "detail_id": detail_id,
        "source": "topcv",
        "source_job_id": source_job_id,
        "job_url": job_url,
        "title_raw": title,
        "company_name_raw": comp_name,
        "company_url": comp_url,
        "company_size_raw": comp_size,
        "company_location_raw": comp_loc,
        "industry_raw": industry,
        "salary_raw": salary,
        "experience_raw": exp,
        "education_raw": edu,
        "location_raw": location,
        "work_mode_raw": work_mode,
        "employment_type_raw": emp_type,
        "working_time_raw": working_time,
        "overview_raw": overview_raw,
        "responsibilities_raw": responsibilities_raw,
        "requirements_raw": requirements_raw,
        "benefits_raw": benefits_raw,
        "required_skills_raw": required_skills_raw,
        "preferred_skills_raw": preferred_skills_raw,
        "published_at_raw": published_at,
        "updated_at_raw": updated_at,
        "deadline_raw": deadline,
        "crawled_at": datetime.now().isoformat()
    }

In [13]:
def crawl_all_job_details(
    input_json_path: str = "../data/raw/topcv_it_jobs_raw.json",
    output_json_path: str = "../data/raw/topcv_job_details_raw.json",
    output_csv_path: str = "../data/raw/topcv_job_details_raw.csv",
    limit: int = None,
    delay_range: tuple = (1.5, 3.0),
    save_interval: int = 10
) -> pd.DataFrame:
    """
    Duyệt qua từng job_url trong danh sách raw và cào thông tin chi tiết đầy đủ.
    Có cơ chế lưu lũy tiến (auto checkpoint) và tự động bỏ qua các job đã cào.
    """
    if not os.path.exists(input_json_path):
        raise FileNotFoundError(f"Không tìm thấy file nguồn: {input_json_path}")

    with open(input_json_path, "r", encoding="utf-8") as f:
        jobs_list = json.load(f)

    if limit:
        jobs_list = jobs_list[:limit]

    # Đọc dữ liệu đã cào trước đó nếu có (Resume checkpoint)
    existing_details = {}
    if os.path.exists(output_json_path):
        try:
            with open(output_json_path, "r", encoding="utf-8") as f:
                old_data = json.load(f)
                for item in old_data:
                    if item.get("detail_id"):
                        existing_details[item["detail_id"]] = item
            print(f"[*] Đã tìm thấy {len(existing_details)} jobs đã cào trước đó. Sẽ tự động bỏ qua.")
        except Exception as e:
            print(f"[!] Không đọc được file cũ, sẽ cào mới: {e}")

    session = requests.Session(impersonate="chrome124")
    crawled_results = list(existing_details.values())
    new_crawled_count = 0

    print(f"[*] Bắt đầu crawl chi tiết cho {len(jobs_list)} jobs...")
    
    for job in tqdm(jobs_list, desc="Crawl Job Details"):
        job_id = str(job.get("source_job_id", "")).strip()
        detail_id = f"topcv_{job_id}"
        
        # Nếu đã cào rồi thì bỏ qua
        if detail_id in existing_details:
            continue

        url = job.get("job_url")
        if not url:
            continue

        max_retries = 3
        success = False
        for attempt in range(max_retries):
            try:
                resp = session.get(url, timeout=20)
                if resp.status_code == 200:
                    detail_data = parse_job_detail(job, resp.text)
                    existing_details[detail_id] = detail_data
                    crawled_results.append(detail_data)
                    new_crawled_count += 1
                    success = True
                    break
                elif resp.status_code == 429:
                    wait_time = 15 * (attempt + 1)
                    print(f"\n[!] Rate limit (429) tại {url}. Nghỉ {wait_time}s rồi thử lại...")
                    time.sleep(wait_time)
                    session = requests.Session(impersonate="chrome124")
                else:
                    print(f"\n[!] Job {job_id} trả về mã {resp.status_code}. Thử lại {attempt + 1}/{max_retries}...")
                    time.sleep(3 * (attempt + 1))
            except Exception as e:
                print(f"\n[!] Lỗi khi cào job {job_id}: {e}")
                time.sleep(3 * (attempt + 1))

        if not success:
            print(f"[X] Bỏ qua job {job_id} sau {max_retries} lần thử thất bại.")

        # Lưu checkpoint định kỳ sau mỗi save_interval jobs mới
        if new_crawled_count > 0 and new_crawled_count % save_interval == 0:
            with open(output_json_path, "w", encoding="utf-8") as f:
                json.dump(crawled_results, f, ensure_ascii=False, indent=2)
            pd.DataFrame(crawled_results).to_csv(output_csv_path, index=False, encoding="utf-8-sig")

        # Nghỉ ngẫu nhiên giữa các request
        time.sleep(random.uniform(*delay_range))

    # Lưu kết quả cuối cùng
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(crawled_results, f, ensure_ascii=False, indent=2)
    df_details = pd.DataFrame(crawled_results)
    df_details.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

    print(f"\n[+] Hoàn tất! Đã lưu {len(df_details)} job details vào:")
    print(f"    - JSON: {output_json_path}")
    print(f"    - CSV:  {output_csv_path}")
    return df_details

In [14]:
# Chạy crawl chi tiết:
# Lưu ý: Có thể đặt limit=5 hoặc limit=10 để test thử trước.
# Khi muốn chạy hết toàn bộ 1900 jobs, hãy đặt limit=None.
df_job_details = crawl_all_job_details(
    input_json_path="../data/raw/topcv_it_jobs_raw.json",
    output_json_path="../data/raw/topcv_job_details_raw.json",
    output_csv_path="../data/raw/topcv_job_details_raw.csv",
    limit=None,            # Đặt limit=10 nếu muốn test trước, hoặc None để cào hết
    delay_range=(1.5, 3.0),# Delay ngẫu nhiên giữa các job tránh bị 429
    save_interval=10       # Tự động lưu checkpoint sau mỗi 10 jobs
)

print("Kích thước DataFrame:", df_job_details.shape)
df_job_details.head(3)

[*] Bắt đầu crawl chi tiết cho 1900 jobs...


Crawl Job Details:   3%|▎         | 62/1900 [03:16<56:52,  1.86s/it]  


[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/business-analyst-ba-3-nam-kinh-nghiem-thu-nhap-hap-dan-tai-ha-noi/2318234.html. Nghỉ 15s rồi thử lại...

[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/business-analyst-ba-3-nam-kinh-nghiem-thu-nhap-hap-dan-tai-ha-noi/2318234.html. Nghỉ 30s rồi thử lại...

[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/business-analyst-ba-3-nam-kinh-nghiem-thu-nhap-hap-dan-tai-ha-noi/2318234.html. Nghỉ 45s rồi thử lại...
[X] Bỏ qua job 2318234 sau 3 lần thử thất bại.


Crawl Job Details:   3%|▎         | 64/1900 [04:53<8:10:18, 16.02s/it]


[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/lap-trinh-fullstack-ha-noi-muc-luong-upto-23-trieu-tu-2-nam-kinh-nghiem/2106213.html. Nghỉ 15s rồi thử lại...

[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/lap-trinh-fullstack-ha-noi-muc-luong-upto-23-trieu-tu-2-nam-kinh-nghiem/2106213.html. Nghỉ 30s rồi thử lại...

[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/lap-trinh-fullstack-ha-noi-muc-luong-upto-23-trieu-tu-2-nam-kinh-nghiem/2106213.html. Nghỉ 45s rồi thử lại...
[X] Bỏ qua job 2106213 sau 3 lần thử thất bại.


...


[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/nhan-vien-tu-van-trien-khai-phan-mem-thu-nhap-den-20tr-duoc-dao-tao/2292668.html. Nghỉ 15s rồi thử lại...


Crawl Job Details:   8%|▊         | 156/1900 [23:05<6:02:27, 12.47s/it]


[!] Rate limit (429) tại https://www.topcv.vn/viec-lam/data-engineer/2296084.html. Nghỉ 15s rồi thử lại...


Crawl Job Details:   8%|▊         | 156/1900 [23:24<4:21:42,  9.00s/it]


KeyboardInterrupt: 